<a id="s11"></a>
# Email Reply Generation — Enron-only (LoRA + Unsloth)
## Two run modes in one notebook — pick ONE path each session

| | **PATH A — Train from scratch** | **PATH B — Load both adapters from zip** |
|---|---|---|
| Use when | You haven't trained yet, or want to retrain | You already have BOTH adapter zips saved from earlier sessions and just want to compare/demo |
| What it does | Downloads Enron, builds dataset, trains a new LoRA adapter | Skips training entirely — you upload two zip files when prompted |
| Time | ~15–70 min (training) | ~2 min (just loading) |

**Run order, either way:**
1. **Part 0 — Shared Setup** (always run this first)
2. **EITHER Part A (train) OR Part B (upload both zips)** — run only ONE of these two parts, not both
3. **Part 3 — Load the OLD adapter** (always run — needed by both paths, since the old adapter always comes from a previous session's zip)
4. **Part 4 — Shared generation function** (always run)
5. **Part 5 — Evaluation** and **Part 6 — App** (always run, in either mode)

> Run in a **fresh Colab runtime** (Runtime → Restart session) before starting either path.

---
# PART 0 — Shared Setup (always run this first, regardless of path)

In [1]:
%pip install -q unsloth
%pip install -q kagglehub evaluate rouge_score gradio
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — enable T4!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [2]:
# ================= CONFIG =================
MODEL_ID = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ  = 1024

# reply-quality filters (only used in PATH A) — starts strict (350), auto-relaxes to clear the floor
MIN_REPLY_LEN, MAX_REPLY_LEN = 350, 1200
MIN_TRAIN_PAIRS = 10000
MIN_INCOMING_LEN, MAX_INCOMING_LEN = 50, 1500
N_EVAL = 100

# LoRA (only used in PATH A)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]

# training (only used in PATH A)
LR          = 2e-4
EPOCHS      = 1
BATCH       = 2
GRAD_ACCUM  = 4        # effective batch = 8

# adapter file/folder names — used by BOTH paths
ADAPTER_DIR     = "email_reply_lora_enron_LONG_adapter"   # NEW (long-reply) adapter
OLD_ADAPTER_DIR = "email_reply_lora_enron_adapter"        # OLD (short-reply-incl.) adapter
NEW_ADAPTER_ZIP = f"{ADAPTER_DIR}.zip"       # used only in PATH B (upload)
OLD_ADAPTER_ZIP = "email_reply_lora_enron_adapter.zip"    # used in PATH A (after training) and PATH B

INTENT_AUG_PROB = 0.5

EMAIL_PROMPT = (
    "You are a professional corporate email assistant. Write a professional, "
    "polite and concise reply to the incoming email.\n"
    "### Subject:\n{subject}\n### Incoming Email:\n{incoming}\n### Reply:\n"
)
INTENT_LINE = "### The reply should convey: {intent}\n### Reply:"
ARTIFACT_SUPPRESS = (
    "\nOutput ONLY the reply text itself. Do not include placeholders such as "
    "[Your Name], [Company Name], or [Position]. Do not add any notes, explanations, "
    "or commentary about the reply. Do not repeat or redraft the email."
)
print("Config loaded.")

Config loaded.


## Load the base model (always run — needed by both paths)

In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ,
    load_in_4bit   = True,
)
print("Base model loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded.


---
# 🅰️ PATH A — Train From Scratch
### Run this whole section ONLY if you have NOT already trained an adapter. Skip straight to PART B below if you already have both adapter zips.

## A.1 · Build the Enron reply-pair dataset (long replies only)
Tries `MIN_REPLY_LEN = 350` first; auto-relaxes in steps of 50 only if needed to keep at least `MIN_TRAIN_PAIRS`.

In [ ]:
import kagglehub, pandas as pd, email as email_lib, re, random
from datasets import Dataset

random.seed(7)

path = kagglehub.dataset_download("wcukierski/enron-email-dataset")
CSV = f"{path}/emails.csv"
print("Dataset at:", CSV)

MARKER  = "-----Original Message-----"
SIGNOFF = re.compile(r"^(thanks|thank you|regards|best|best regards|cheers|thx)\b[.,!]?$", re.I)

def clean_original(text):
    lines = []
    for l in text.strip().splitlines():
        l = re.sub(r"^\s*>+\s*", "", l)
        if re.match(r"^\s*(From|Sent|To|Cc|Subject)\s*:", l):
            continue
        l = re.sub(r"\[mailto:[^]]+\]", "", l)
        lines.append(l)
    return " ".join(" ".join(lines).split())

def clean_reply(text):
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    while lines and (len(lines[-1].split()) <= 3 or SIGNOFF.match(lines[-1])):
        lines.pop()
    return " ".join(" ".join(lines).split())

def extract_pairs(chunk):
    out = []
    for raw in chunk["message"]:
        try:
            msg  = email_lib.message_from_string(raw)
            body = msg.get_payload()
            if not isinstance(body, str) or MARKER not in body:
                continue
            reply, original = body.split(MARKER, 1)
            original, reply = clean_original(original), clean_reply(reply)
            subject = re.sub(r"^\s*(RE|FW|FWD)\s*:\s*", "", (msg.get("Subject") or "").strip(), flags=re.I)
            if (MIN_REPLY_LEN <= len(reply) <= MAX_REPLY_LEN
                    and MIN_INCOMING_LEN <= len(original) <= MAX_INCOMING_LEN
                    and "forwarded by" not in reply.lower()):
                out.append({"subject": subject, "incoming": original, "reply": reply})
        except Exception:
            pass
    return out

def extract_all_pairs(min_reply_len):
    global MIN_REPLY_LEN
    MIN_REPLY_LEN = min_reply_len
    collected = []
    for chunk in pd.read_csv(CSV, chunksize=20000):
        collected += extract_pairs(chunk)
    return collected

threshold = MIN_REPLY_LEN
pairs = extract_all_pairs(threshold)
print(f"MIN_REPLY_LEN={threshold} → {len(pairs)} pairs")

while len(pairs) - N_EVAL < MIN_TRAIN_PAIRS and threshold > 100:
    threshold -= 50
    pairs = extract_all_pairs(threshold)
    print(f"  relaxed to MIN_REPLY_LEN={threshold} → {len(pairs)} pairs")

MIN_REPLY_LEN = threshold
random.shuffle(pairs)
print(f"Total usable pairs: {len(pairs)}")

train_pairs, eval_pairs = pairs[N_EVAL:], pairs[:N_EVAL]
print(f"Train: {len(train_pairs)}   Eval (held-out): {len(eval_pairs)}")

In [ ]:
# ---------------- keyword-bag intent augmentation (fixed version) ----------------
STOPWORDS = set("the a an and or but if of to in on at for with from is are was were be been being "
                "this that these those i you we they he she it will would can could should shall".split())

def pseudo_intent(reply: str) -> str:
    words = re.findall(r"[a-zA-Z']{4,}", reply.lower())
    kw, seen = [], set()
    for w in words:
        if w not in STOPWORDS and w not in seen:
            kw.append(w); seen.add(w)
        if len(kw) == 5:
            break
    random.shuffle(kw)
    return "mention " + ", ".join(kw) if kw else "acknowledge the message"

def to_training_text(p):
    prompt = EMAIL_PROMPT.format(subject=p["subject"], incoming=p["incoming"])
    if random.random() < INTENT_AUG_PROB:
        prompt = prompt.replace("### Reply:", INTENT_LINE.format(intent=pseudo_intent(p["reply"])))
    return {"text": prompt + p["reply"] + tokenizer.eos_token}

email_train = Dataset.from_list([to_training_text(p) for p in train_pairs])
print(email_train)
print("-"*70)
print(email_train[0]["text"][:800])

## A.2 · Apply LoRA (this becomes the "default" / New adapter)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGETS, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
model.print_trainable_parameters()

## A.3 · Train

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

kw = dict(
    output_dir="qwen15b-email-lora-enron-long",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=20,
    save_steps=200,
    optim="adamw_8bit",
    report_to="none",
    dataset_text_field="text",
)
try:
    cfg = SFTConfig(max_seq_length=MAX_SEQ, **kw)
except TypeError:
    cfg = SFTConfig(max_length=MAX_SEQ, **kw)
try:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, args=cfg, train_dataset=email_train)
except TypeError:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, args=cfg, train_dataset=email_train)

res = trainer.train()
print(f"Done — {res.metrics['train_runtime']/60:.1f} min, final loss {res.metrics['train_loss']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

hist = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
s, l = zip(*hist)
plt.figure(figsize=(7,3.5)); plt.plot(s, l, color="#C67512", linewidth=2)
plt.xlabel("step"); plt.ylabel("training loss"); plt.title("Email-reply LoRA (long-reply, Enron-only) — training loss")
plt.grid(alpha=.3); plt.savefig("email_loss_long.png", dpi=150, bbox_inches="tight"); plt.show()

## A.4 · Save and download the new adapter (as a backup zip for future sessions)

In [ ]:
import shutil, os
from google.colab import files

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
mb = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f)) for f in os.listdir(ADAPTER_DIR)) / 1e6
print(f"New adapter saved to '{ADAPTER_DIR}' ({mb:.1f} MB)")

shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)
files.download(f"{ADAPTER_DIR}.zip")
print("Downloading new adapter as backup — keep this zip for PATH B in future sessions.")

**PATH A complete.** The trained model is in memory with the NEW adapter active (named `"default"`). Now skip PART B below and go straight to **PART 3 — Load the OLD adapter**.

---
# 🅱️ PATH B — Load Both Adapters From Zip (skip training)
### Run this whole section ONLY if you already have BOTH adapter zip files saved from previous sessions, and want to skip training entirely.

## B.1 · Upload and unzip the NEW adapter

In [4]:
import os, zipfile
from google.colab import files

if not os.path.exists(NEW_ADAPTER_ZIP):
    print(f"'{NEW_ADAPTER_ZIP}' not found — please upload it now (the NEW / long-reply adapter):")
    uploaded = files.upload()
    NEW_ADAPTER_ZIP = list(uploaded.keys())[0]
print(f"Using NEW adapter zip: {NEW_ADAPTER_ZIP}")

if not os.path.exists(ADAPTER_DIR):
    os.makedirs(ADAPTER_DIR)
with zipfile.ZipFile(NEW_ADAPTER_ZIP, "r") as zip_ref:
    zip_ref.extractall(ADAPTER_DIR)
print(f"Unzipped NEW adapter to '{ADAPTER_DIR}'")

Using NEW adapter zip: email_reply_lora_enron_LONG_adapter.zip
Unzipped NEW adapter to 'email_reply_lora_enron_LONG_adapter'


## B.2 · Attach the NEW adapter to the base model
Since no training happened in this session, `model` is still a plain base model — this step attaches the NEW adapter and turns `model` into a PEFT (LoRA-aware) model, named `"default"`.

In [5]:
from peft import PeftModel

model = PeftModel.from_pretrained(model, ADAPTER_DIR, adapter_name="default")
model.set_adapter("default")
print("NEW adapter attached and active as 'default'.")

NEW adapter attached and active as 'default'.


**PATH B complete (for the new adapter).** Now go to **PART 3 — Load the OLD adapter** below (this uploads and attaches the second zip).

---
# PART 3 — Load the OLD Adapter (always run — needed by BOTH paths)
The old adapter always comes from a previous session's zip, regardless of which path you took above.

In [6]:
import os, zipfile
from google.colab import files

if not os.path.exists(OLD_ADAPTER_ZIP):
    print(f"'{OLD_ADAPTER_ZIP}' not found — please upload it now (the OLD / short-reply-incl. adapter):")
    uploaded = files.upload()
    OLD_ADAPTER_ZIP = list(uploaded.keys())[0]
print(f"Using OLD adapter zip: {OLD_ADAPTER_ZIP}")

if not os.path.exists(OLD_ADAPTER_DIR):
    os.makedirs(OLD_ADAPTER_DIR)
with zipfile.ZipFile(OLD_ADAPTER_ZIP, "r") as zip_ref:
    zip_ref.extractall(OLD_ADAPTER_DIR)
print(f"Unzipped OLD adapter to '{OLD_ADAPTER_DIR}'")

Using OLD adapter zip: email_reply_lora_enron_adapter.zip
Unzipped OLD adapter to 'email_reply_lora_enron_adapter'


In [7]:
# Attach the OLD adapter to the SAME model (already carrying the NEW adapter as "default")
if "old_short" not in model.peft_config:
    model.load_adapter(OLD_ADAPTER_DIR, adapter_name="old_short")
    print("Loaded 'old_short' adapter.")
else:
    print("'old_short' adapter already loaded.")

model.set_adapter("default")   # make sure the NEW adapter is the active one by default
print("Both adapters now loaded on one model: 'default' (new, long-reply) and 'old_short' (old).")

Loaded 'old_short' adapter.
Both adapters now loaded on one model: 'default' (new, long-reply) and 'old_short' (old).


---
# PART 4 — Shared Generation Function (always run)
Used by BOTH the evaluation and the app below — greedy decoding everywhere, so what gets scored is exactly what the demo shows.

In [8]:
import re as _re
FastLanguageModel.for_inference(model)

def generate_reply(subject, incoming, intent="", use_adapter=True, suppress_artifacts=False,
                   adapter_name="default", max_new_tokens=180):
    prompt = EMAIL_PROMPT.format(subject=subject, incoming=incoming[:1500])
    if intent.strip():
        prompt = prompt.replace("### Reply:", INTENT_LINE.format(intent=intent.strip()))
    if suppress_artifacts and not use_adapter:
        prompt = prompt.replace("### Reply:", ARTIFACT_SUPPRESS + "\n### Reply:")

    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    if use_adapter:
        model.set_adapter(adapter_name)
    ctx = model.disable_adapter() if not use_adapter else torch.no_grad()
    with ctx, torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, min_new_tokens=25,
                             do_sample=False, repetition_penalty=1.1, no_repeat_ngram_size=3,
                             pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    reply = text.split("### Reply:")[-1].strip()

    cut = _re.search(r"\n\s*(Subject\s*:|Incoming Email\s*:|###|Note\s*:)", reply)
    if cut:
        reply = reply[:cut.start()].strip()
    angle_email = _re.search(r"<[\w.\-]+@[\w.\-]+>", reply)
    if angle_email:
        reply = reply[:angle_email.start()].strip()
    greet = list(_re.finditer(r"\b(Dear\s+\w+|Hi\s+\w+|Hello\s+\w+)\b[,.]", reply, _re.I))
    if len(greet) > 1:
        reply = reply[:greet[1].start()].strip()
    return reply

demo_q = ("Meeting reschedule request",
          "Hi, can we move our project review from Thursday to next week? Some reports are still being finalized.")
print("Sanity check:")
print("BASE (clean)   :", generate_reply(*demo_q, use_adapter=False, suppress_artifacts=True)[:200])
print("LoRA (new)     :", generate_reply(*demo_q, use_adapter=True, adapter_name="default",
      intent="agree to reschedule, propose Tuesday 10 AM")[:200])
print("LoRA (old)     :", generate_reply(*demo_q, use_adapter=True, adapter_name="old_short",
      intent="agree to reschedule, propose Tuesday 10 AM")[:200])

Sanity check:


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

BASE (clean)   : Of course! We can definitely reschedule the meeting for next week. Please let me know your preferred date and time. Thank you.


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA (new)     : Agree to rescheduling for Tuesday at 12 noon. I will send you an e-mail with the details of the meeting. Thanks! "Kumar, Sujit"
LoRA (old)     : Agree to rescheduling for Tuesday at 12 noon. I will send you an e-mail with the details of the meeting. Thanks. From: Kay Mann/ENRON@enronXgate on 05/14/2001 09:37 AM To: John D Williams/HOU/EES@EES 


---
# PART 5 — Evaluation — Base vs LoRA (New) vs LoRA (Old)
> **Note:** this section needs `eval_pairs`, which only exists if you ran **PATH A** in this session (it builds the held-out set from the dataset). If you ran **PATH B only** (no training, just loaded both adapters), skip this section and go straight to **PART 6 — App** — you can still demo and compare live, just without a scored table.

In [ ]:
import evaluate
from tqdm.auto import tqdm

rouge = evaluate.load("rouge")
preds = {"base_clean": [], "lora_new": [], "lora_old": []}
refs  = []
qualitative = []

for p in tqdm(eval_pairs, desc="evaluating"):
    refs.append(p["reply"])
    bc = generate_reply(p["subject"], p["incoming"], use_adapter=False, suppress_artifacts=True)
    ln = generate_reply(p["subject"], p["incoming"], use_adapter=True, adapter_name="default")
    lo = generate_reply(p["subject"], p["incoming"], use_adapter=True, adapter_name="old_short")
    preds["base_clean"].append(bc)
    preds["lora_new"].append(ln)
    preds["lora_old"].append(lo)
    if len(qualitative) < 6:
        qualitative.append((p["subject"], p["incoming"], p["reply"], bc, ln, lo))

print(f"\n{'variant':14}{'ROUGE-1':>10}{'ROUGE-L':>10}{'avg words':>12}")
for name in ("base_clean", "lora_new", "lora_old"):
    r = rouge.compute(predictions=preds[name], references=refs)
    avg_len = sum(len(x.split()) for x in preds[name]) / len(preds[name])
    print(f"{name:14}{r['rouge1']:>10.3f}{r['rougeL']:>10.3f}{avg_len:>12.1f}")

ref_len = sum(len(x.split()) for x in refs) / len(refs)
print(f"\n(reference average length: {ref_len:.1f} words)")

In [ ]:
# qualitative side-by-side (screenshot these for the report)
for subj, inc, ref, bc, ln, lo in qualitative[:4]:
    print("="*90)
    print(f"SUBJECT : {subj}")
    print(f"INCOMING: {inc[:200]}...")
    print(f"\nREFERENCE       : {ref[:260]}")
    print(f"BASE (clean)    : {bc[:260]}")
    print(f"LoRA (new)      : {ln[:260]}")
    print(f"LoRA (old)      : {lo[:260]}")

---
# PART 6 — Web App — Base vs LoRA (New) vs LoRA (Old), side by side
Works after EITHER path, as long as PART 3 (old adapter) and PART 4 (generation function) have both run.

In [9]:
import gradio as gr

def compare_all_models(subject, incoming, intent):
    base_clean_reply = generate_reply(subject, incoming, use_adapter=False, suppress_artifacts=True)
    new_lora_reply   = generate_reply(subject, incoming, intent, use_adapter=True, adapter_name="default")
    old_lora_reply   = generate_reply(subject, incoming, intent, use_adapter=True, adapter_name="old_short")
    return base_clean_reply, new_lora_reply, old_lora_reply

with gr.Blocks(title="Email Reply Assistant — Base vs New vs Old LoRA (Enron)") as app:
    gr.Markdown(
        "## Professional Email Reply Generator — Base vs New LoRA vs Old LoRA (Enron-only)\n"
        "*New LoRA: trained on longer, non-terse replies. Old LoRA: original short-reply-inclusive training.*"
    )
    subj = gr.Textbox(label="Subject", value="Meeting reschedule request")
    body = gr.Textbox(label="Incoming email", lines=6, value=(
        "Hi, I wanted to check if we could move our project review meeting from Thursday "
        "to early next week. A few of the reports are still being finalized."))
    intent = gr.Textbox(label="What should the reply say? (optional)",
                        placeholder="e.g. agree to reschedule, propose Tuesday 10 AM")
    btn = gr.Button("Generate Replies", variant="primary")
    with gr.Row():
        ob = gr.Textbox(label="Base model reply (clean)", lines=10)
        ol_new = gr.Textbox(label="LoRA (New — long replies)", lines=10)
        ol_old = gr.Textbox(label="LoRA (Old — short replies incl.)", lines=10)
    btn.click(compare_all_models, [subj, body, intent], [ob, ol_new, ol_old])
    gr.Examples([
        ["Meeting reschedule request",
         "Hi, can we move our project review from Thursday to next week? Some reports are still being finalized.",
         "agree to reschedule, propose Tuesday 10 AM"],
        ["Status update request",
         "Could you give me a quick update on where we stand with the contract review?",
         "review is nearly done, summary will be sent Thursday"],
        ["Document request",
         "Please send me the latest version of the operating statement when you get a chance.", ""],
    ], [subj, body, intent])

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ff06ab841bff32ec06.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Sample Inputs:
Set 1 — Clean, safe demo examples (use these first, most likely to look good)

1. Meeting reschedule

Subject: Project review meeting

Incoming: Hi, can we move our project review from Thursday to next week? Some of the reports are still being finalized and I'd rather we go in with complete numbers.

Intent: agree to reschedule, propose Tuesday 10 AM

2. Status check-in

Subject: Status update request

Incoming: Could you give me a quick update on where we stand with the contract review? I need to report back to the team by Friday.

Intent: review is almost done, summary will be sent by Thursday

3. Simple document request (no intent — tests the plain path)

Subject: Document request

Incoming: Please send me the latest version of the operating statement when you get a chance.

Intent: (leave blank)

Set 2 — Slightly harder, good for showing nuance in your report

4. Internal approval / sign-off

Subject: Approval needed

Incoming: Can you approve the attached expense report before end of day? I need it processed before the month closes.

Intent: approve it, mention it will be processed today


5. Short, blunt real-Enron-style input (tests whether LoRA matches that terseness)

Subject: (leave blank)

Incoming: Any update on the numbers?

Intent: (leave blank)
What to deliberately test and show as a "known limitation" (don't hide this — use it)

6. Out-of-domain (should misfire — this is your honest limitation slide)

Subject: Course registration query

Incoming: Could you let me know the deadline for registering for next semester's electives?

Intent: (leave blank)


---
## Notes
- **PATH A** and **PATH B** both end with `model` holding two named adapters: `"default"` (new) and `"old_short"` (old) — everything after PART 3 works identically regardless of which path you took.
- **PART 5 (scored evaluation)** only works after PATH A, since it needs the held-out `eval_pairs` built during dataset preparation. After PATH B, go straight to PART 6 for a live, unscored demo.
- Only ever run ONE of PATH A / PATH B per session — running both would waste time (Path A already gives you a trained "default" adapter; Path B is only needed when you don't want to retrain).